In [7]:
import pandas as pd
import numpy as np

def answer_one():
    # 1. Energy Indicators
    energy = pd.read_excel('D:/Energy Indicators.xls', skiprows=17, skipfooter=38, engine='xlrd')
    energy = energy.iloc[:, 2:]
    energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']
    energy.replace('...', np.nan, inplace=True)
    energy['Energy Supply'] = pd.to_numeric(energy['Energy Supply']) * 1000000
    
    def clean_country(data):
        data = ''.join([i for i in data if not i.isdigit()])
        if '(' in data: data = data.split('(')[0]
        return data.strip()
    
    energy['Country'] = energy['Country'].apply(clean_country)
    energy['Country'] = energy['Country'].replace({
        "Republic of Korea": "South Korea",
        "United States of America": "United States",
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
        "China, Hong Kong Special Administrative Region": "Hong Kong"
    })

    # 2. World Bank GDP (Твій робочий метод!)
    raw_gdp = pd.read_excel('D:/world_bank.xls', sheet_name='Data', engine='xlrd')
    header_row = 0
    for i in range(len(raw_gdp)):
        if "Country Name" in raw_gdp.iloc[i].values or "Country" in raw_gdp.iloc[i].values:
            header_row = i + 1
            break
    gdp = pd.read_excel('D:/world_bank.xls', sheet_name='Data', skiprows=header_row, engine='xlrd')
    gdp.rename(columns={gdp.columns[0]: 'Country'}, inplace=True)
    gdp['Country'] = gdp['Country'].replace({
        "Korea, Rep.": "South Korea", 
        "Iran, Islamic Rep.": "Iran", 
        "Hong Kong SAR, China": "Hong Kong"
    })
    gdp.columns = [str(c).split('.')[0] for c in gdp.columns]
    years = [str(y) for y in range(2006, 2016)]
    GDP_subset = gdp[['Country'] + years]

    # 3. ScimEn
    ScimEn = pd.read_excel('D:/scimagojr.xlsx', engine='openpyxl')
    ScimEn_top15 = ScimEn[:15]

    # 4. Merging
    # Спочатку ScimEn та Energy
    df = pd.merge(ScimEn_top15, energy, how='inner', on='Country')
    # Потім додаємо ВВП
    final_df = pd.merge(df, GDP_subset, how='inner', on='Country')
    
    return final_df.set_index('Country')

# Перевірка результату
try:
    result = answer_one()
    print("🎉 Фінальна таблиця готова!")
    print(f"Розмір: {result.shape} (має бути 15 рядочків на 20 колонок)")
    display(result.head(3))
except Exception as e:
    print(f"❌ Помилка при об'єднанні: {e}")

🎉 Фінальна таблиця готова!
Розмір: (15, 21) (має бути 15 рядочків на 20 колонок)


,Rank,Region,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,,
China,1,Asiatic Region,273437,272374,2336764,1615239,8.55,245,1.271910e+11,93,...,2.752119e+12,3.550328e+12,4.594337e+12,5.101691e+12,6.087192e+12,7.551545e+12,8.532186e+12,9.570471e+12,1.047562e+13,1.106157e+13
United States,2,Northern America,175891,172431,2230544,724472,12.68,363,9.083800e+10,286,...,1.381559e+13,1.447423e+13,1.476986e+13,1.447806e+13,1.504896e+13,1.559973e+13,1.625397e+13,1.684319e+13,1.755068e+13,1.820602e+13
India,3,Asiatic Region,55082,53775,463165,162944,8.41,181,3.319500e+10,26,...,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856721e+12,2.039126e+12,2.103588e+12
